# Argus: train YOLOv8 on VisDrone (Colab)

End-to-end on a Colab GPU runtime: install, fetch VisDrone, fine-tune
YOLOv8s at 1280 px, evaluate mAP, and export an INT8 TensorRT engine.

Set the runtime to **GPU** before running (Runtime > Change runtime type).

In [ ]:
!nvidia-smi
!git clone https://github.com/softwarelijah/argus.git
%cd argus
!pip install -q -e ".[train,export]"

In [ ]:
# Fetch and convert VisDrone to YOLO format.
# If Google Drive rate-limits, upload the zips to datasets/VisDrone and add --skip-download.
!python scripts/download_visdrone.py --root datasets/VisDrone
!python scripts/prepare_visdrone.py --root datasets/VisDrone

In [ ]:
# Fine-tune. Drop batch to 8 if you hit OOM at 1280 px.
!python scripts/train.py --config configs/train.yaml

In [ ]:
!python scripts/evaluate.py --weights runs/visdrone/yolov8s-1280/weights/best.pt

In [ ]:
# Export INT8 TensorRT engine (build on the same GPU arch you deploy to).
!pip install -q tensorrt pycuda
!python scripts/export_tensorrt.py \
    --weights runs/visdrone/yolov8s-1280/weights/best.pt \
    --calib-images datasets/VisDrone/VisDrone2019-DET-val/images \
    --precision int8

In [ ]:
# Benchmark FPS / latency (synthetic frames, no video file needed).
!python scripts/benchmark.py --synthetic 300 \
    --engine weights/best-int8.engine --imgsz 1280